In [ ]:
!pip install yt-dlp librosa

In [ ]:
import yt_dlp
import librosa
import numpy as np
import pandas as pd
import tempfile
import os
import time
import glob
import shutil
from google.colab import drive

In [ ]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def get_audio(song_name, artist):
    query = f"{song_name} {artist} official audio"
    with tempfile.TemporaryDirectory() as tmpdir:
        ydl_opts = {
            'format': 'bestaudio/best',
            'quiet': True,
            'noplaylist': True,
            'outtmpl': os.path.join(tmpdir, 'audio.%(ext)s'),
            'postprocessors': [{
                'key': 'FFmpegExtractAudio',
                'preferredcodec': 'mp3',
            }],
        }
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([f"ytsearch1:{query}"])
        audio_file = os.path.join(tmpdir, 'audio.mp3')
        y, sr = librosa.load(audio_file)
    return y, sr

In [ ]:
def extract_features(y, sr):
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)

    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    mfccs_mean = np.mean(mfccs, axis=1)

    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    chroma_mean = np.mean(chroma, axis=1)
    chroma_std = np.std(chroma, axis=1)

    spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
    spectral_centroid_mean = np.mean(spectral_centroid)

    features = {"tempo": float(tempo[0])}
    for i, v in enumerate(mfccs_mean):
        features[f"mfcc_{i+1}"] = round(float(v), 4)
    for i, v in enumerate(chroma_mean):
        features[f"chroma_mean_{i+1}"] = round(float(v), 4)
    for i, v in enumerate(chroma_std):
        features[f"chroma_std_{i+1}"] = round(float(v), 4)
    features["spectral_centroid"] = round(float(spectral_centroid_mean), 4)

    return features

In [ ]:
def process_songs(csv_path, output_path):
    df = pd.read_csv(csv_path)
    results = []

    for idx, row in df.iterrows():
        song = row['Song Name']
        artist = row['Artist Name']
        print(f"Processing {idx+1}/{len(df)}: {song} - {artist}")

        try:
            y, sr = get_audio(song, artist)
            features = extract_features(y, sr)
            features['song_title'] = song
            features['artist'] = artist
            results.append(features)
        except Exception as e:
            print(f"  Failed: {e}")
            continue

    output_df = pd.DataFrame(results)
    cols = ['song_title', 'artist'] + [c for c in output_df.columns if c not in ['song_title', 'artist']]
    output_df = output_df[cols]
    output_df.to_csv(output_path, index=False)
    print(f"Done! Saved {len(results)} songs to {output_path}")

In [ ]:
# ==============================
# CONFIGURE YOUR YEARS HERE
# ==============================
YEARS_TO_PROCESS = [2000,2001]  # Change this to whatever years you want
# ==============================

def process_all_songs(years, input_folder, output_folder):
    print(f"Processing {len(years)} years: {years}")

    for year in years:
        csv_path = f'{input_folder}/Billboard_Top_100_{year}.csv'
        output_path = f'{output_folder}/features_{year}.csv'

        if not os.path.exists(csv_path):
            print(f"Skipping {year} - CSV not found")
            continue

        if os.path.exists(output_path):
            print(f"Skipping {year} - already processed")
            continue

        print(f"\n--- Processing {year} ---")
        start = time.time()
        process_songs(csv_path, output_path)
        elapsed = round((time.time() - start) / 60, 2)
        print(f"Year {year} done in {elapsed} minutes")

process_all_songs(YEARS_TO_PROCESS, '/content', '/content/drive/MyDrive/CLARIFY')